<a href="https://colab.research.google.com/github/rakshitshah280701/InstructAware/blob/main/InstructAware_MetricColab_Bluert_Meteor_USE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Metrics (BLUERT, USE Similarity, METEOR) on Testing Results

---

### Description of Evaluation Script

The script is designed to **evaluate narrative generation quality** using multiple metrics by comparing **predicted narratives** against the **original ground truth narratives**. It accepts as input a **CSV file** that includes the following fields:

- **Image identifier or data reference**
- **Associated vision data**
- **Original narrative (reference)**
- **Predicted narrative (generated by a model)**

Once the CSV is loaded, the script computes three widely-used **natural language evaluation metrics** for each sample:

- **BLEURT**
- **USE Similarity (Universal Sentence Encoder)**
- **METEOR**

After computing these metrics, the script appends **three new columns** to the original CSV file—one for each metric—and saves the augmented file as a new CSV. This allows researchers or developers to evaluate and track model performance at a fine-grained level.

---

## Evaluation Metrics

### 1. **BLEURT**

BLEURT (Bilingual Evaluation Understudy with Representations from Transformers) is a **learned evaluation metric** that compares predicted and reference text using contextual embeddings from a pretrained transformer model fine-tuned for text generation evaluation. Unlike traditional metrics that rely on exact word overlap (like BLEU or ROUGE), BLEURT is trained to **capture semantic similarity**, **grammaticality**, and **meaning preservation** based on human judgments.

- **Why use BLEURT?**  
  BLEURT is especially valuable when minor variations in phrasing don’t significantly affect meaning—e.g., “The car was damaged on the left side” vs. “There was damage to the car’s left side.”
- **Output**: A real-valued score typically between 0 and 1, with higher values indicating closer alignment with the reference narrative.

---

### 2. **USE Similarity (Universal Sentence Encoder)**

The Universal Sentence Encoder (USE) provides **dense vector representations of entire sentences**, capturing both syntactic and semantic information. To evaluate the similarity between two narratives, the script encodes both the **original** and **predicted** narratives into vectors and computes their **cosine similarity**.

- **Why use USE?**  
  USE similarity offers a **simple yet effective** way to measure how semantically close two narratives are, regardless of word choice. It’s less sensitive to word order or token-level overlap and is good for gauging overall content alignment.
- **Output**: A cosine similarity score between 0 (completely different) and 1 (identical in embedding space).

---

### 3. **METEOR**

METEOR (Metric for Evaluation of Translation with Explicit ORdering) is a traditional NLP metric that evaluates generated text by aligning it with the reference text using several levels of matching:

- Exact word matches
- Stem matches
- Synonym matches (using WordNet)
- Paraphrase matches (in extended versions)

It also applies a penalty based on word order to ensure coherent sentence structure is respected.

- **Why use METEOR?**  
  METEOR tends to correlate better with human judgment than BLEU, especially in single-sentence or narrative evaluation, because of its **semantic matching** and **flexible alignment** capabilities.
- **Output**: A score between 0 and 1, with higher values indicating better alignment with the reference narrative.



Got it! Here's an expanded version of your description, followed by a detailed breakdown of each evaluation metric in its own subsection:

---



---




## 📊 Comparison of Evaluation Metrics

| Metric     | Type                  | Captures Semantics? | Handles Synonyms? | Word Order Sensitivity | Output Range | Best For |
|------------|-----------------------|----------------------|--------------------|-------------------------|--------------|----------|
| **BLEURT** | Transformer-based     | ✅ Yes               | ✅ Yes             | ✅ Yes                  | ~0 to 1      | Overall semantic quality, grammar, and fluency |
| **USE**    | Embedding Similarity  | ✅ Yes               | ❌ No              | ❌ No                   | 0 to 1       | Quick semantic similarity check |
| **METEOR** | Rule-based with NLP   | ⚠️ Partial          | ✅ Yes             | ✅ Yes                  | 0 to 1       | Lexical and structural similarity with synonym support |




In [ ]:
!git clone https://github.com/google-research/bleurt.git
%cd bleurt
!pip install .
!wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
!unzip BLEURT-20.zip
!pip install datasets
!pip install tensorflow tensorflow-text
!pip install pandas scikit-learn
!pip install torch torchvision torchaudio

Cloning into 'bleurt'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 134 (delta 0), reused 17 (delta 0), pack-reused 116 (from 1)
Receiving objects: 100% (134/134), 31.28 MiB | 19.93 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/bleurt
Processing /content/bleurt
  Preparing metadata (setup.py) ... done
  Created wheel for BLEURT: filename=BLEURT-0.0.2-py3-none-any.whl size=16456766 sha256=9d9406bf9f54b9d63b9de2fad3b7db6896e5ee429d2168bae8b68138dc78210d
  Stored in directory: /tmp/pip-ephem-wheel-cache-liqudg8m/wheels/49/ab/73/9318ab38d4cd1c732bcea8335d3f8d7c0316c8d07b9084fa85
Successfully built BLEURT
--2025-04-07 20:45:13--  https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.141.207, 74.125.137.207, 142.250.101.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.141

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')  # Optional but sometimes required

# Also, try downloading 'punkt_tab' directly if needed
try:
    nltk.download('punkt_tab')
except:
    print("punkt_tab not found, proceeding without it.")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import tensorflow_hub as hub
import tensorflow as tf
from bleurt import score as bleurt_score
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize



# Load CSV file
# csv_file_path = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5_withFixed_TrainTestSplit/Predicted_Narratives_T5model.csv"
csv_file_path = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel.csv"
predictions_df = pd.read_csv(csv_file_path)

# Fill NaN values to avoid errors
predictions_df['ORIGINAL OUTPUT TEXT'] = predictions_df['ORIGINAL OUTPUT TEXT'].fillna(" ").astype(str)
predictions_df['PREDICTED OUTPUT TEXT'] = predictions_df['PREDICTED OUTPUT TEXT'].fillna(" ").astype(str)

# Initialize BLEURT scorer
bleurt_checkpoint = "BLEURT-20"
bleurt_scorer = bleurt_score.BleurtScorer(bleurt_checkpoint)

# Compute BLEURT Scores
predictions_df['bleurt_score'] = bleurt_scorer.score(
    references=predictions_df['ORIGINAL OUTPUT TEXT'].tolist(),
    candidates=predictions_df['PREDICTED OUTPUT TEXT'].tolist()
)

# Compute METEOR Scores
predictions_df['meteor_score'] = predictions_df.apply(
    lambda row: meteor_score([word_tokenize(row['ORIGINAL OUTPUT TEXT'])], word_tokenize(row['PREDICTED OUTPUT TEXT'])),
    axis=1
)

# Load Universal Sentence Encoder (USE)
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

def compute_use_similarity(row):
    embeddings = use_model([row['PREDICTED OUTPUT TEXT'], row['ORIGINAL OUTPUT TEXT']])
    emb1, emb2 = np.array(embeddings[0]), np.array(embeddings[1])
    return 1 - np.linalg.norm(emb1 - emb2)

# Compute USE Similarity
predictions_df['use_similarity'] = predictions_df.apply(compute_use_similarity, axis=1)

# Save output
output_csv_path = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel_Bluert_Meteor_USE.csv"
predictions_df.to_csv(output_csv_path, index=False)
print(f"✅ Predictions with TensorFlow metrics saved to {output_csv_path}")


✅ Predictions with TensorFlow metrics saved to /content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel_Bluert_Meteor_USE.csv


In [ ]:
import pandas as pd

# Load the CSV file
csv_path = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel_Bluert_Meteor_USE.csv"  # Update the path if needed
df = pd.read_csv(csv_path)

# Display first few rows
df.head()

from google.colab import data_table

# Display CSV as an interactive table
data_table.DataTable(df)

,INPUT TEXT,ORIGINAL OUTPUT TEXT,PREDICTED OUTPUT TEXT,bleurt_score,meteor_score,use_similarity
0,- 'ONE 30 HOUR MINUTE PARKING PARKING 8A.M.-7P...,Look to your left for a parking sign that allo...,"In the center of your view, you can see a sign...",0.640192,0.482624,0.266756
1,- 'KANSAI SUSHI' appears at coordinates [0.619...,"You are near three shops: ""MUSIC DEPOT"" on you...","Right in front of you, there's a sushi restaur...",0.585770,0.296618,0.201577
2,- 'WHEN WORKFEELS OVERWHELMING. REMEMBER YOURE...,You’re facing a big sign that shares a quirky ...,You’re near a sign that reminds you to take ca...,0.361127,0.269102,-0.031564
3,- 'STUDIOKRONER' appears at coordinates [0.652...,"You are near ""STUDIOKRONER"" on your right, whi...","You’re near a shop called StudioKroner, where ...",0.610059,0.379078,0.257413
4,- 'LUCKYSTONE JEWELRY CRAFT' appears at coordi...,You are near several shops and services. To yo...,"You are near a jewelry store called ""LUCKYSTON...",0.615173,0.558801,0.369096
...,...,...,...,...,...,...
166,- 'WRONG WAY' appears at coordinates [0.819444...,"You have a ""WRONG WAY"" sign to your right, whi...","You’re near a traffic sign that reads ""WRONG W...",0.691642,0.311872,0.200964
167,- 'A&W ALL AMERICAN FOOD' appears at coordinat...,"Right in front of you, there's an A&W restaura...","You’re near a fast-food restaurant called ""A&W...",0.696446,0.423731,0.137225
168,- 'WARD POLYMERS LIMITED' appears at coordinat...,"You’re facing a sign that says ""WARD POLYMERS ...","You are near ""WARD POLYMERS LIMITED,"" a sign t...",0.622631,0.447238,0.071750
169,- 'GOLDEN DRAGON DINING TAKE OUT' appears at c...,"Just to your left, there's a big sign for a pl...","To your right, there's a prominent sign for ""G...",0.693138,0.533798,0.190807
